In [ ]:
import os
import mysql.connector
from mysql.connector import Error
#Testing to see if the database is connected succesfully
try:
    connection = mysql.connector.connect(
        host=os.getenv("MYSQL_HOST", "localhost"),
        user=os.getenv("MYSQL_USER", "root"),
        password=os.environ["MYSQL_PASSWORD"],   
        database="wonderland_elections_db",
        port=3306,
    
    )
    if connection.is_connected():
        print("Successfully connected to the Wonderland Elections database!")
except Error as e:
    print("Error while connecting to MySQL:", e)


In [ ]:
import os
import mysql.connector
from mysql.connector import Error
# Connecting and disconnecting
def connect_db():
    """Connect to the Wonderland Elections database."""
    try:
        conn = mysql.connector.connect(
            host=os.getenv("MYSQL_HOST", "localhost"),
            user=os.getenv("MYSQL_USER", "root"),
            password=os.environ["MYSQL_PASSWORD"],  
            database="wonderland_elections_db",
            port=3306
        )
        if conn.is_connected():
            print(" You are Connected to Wonderland Elections DB.")
            return conn
    except Error as e:
        print(" The Database connection failed:", e)
        return None

def close_db(conn):
    """Safely close the database connection."""
    if conn and conn.is_connected():
        conn.close()
        print("The Connection closed.")


In [ ]:
# Test functions
conn = connect_db()  
close_db(conn)        


In [ ]:
def execute_query(query, fetch=True):

#Sending SQL query and retrieving results.
    
    """Execute an SQL statement and optionally return results."""
    conn = connect_db()
    if not conn:
        return None
    try:
        cursor = conn.cursor(dictionary=True)
        cursor.execute(query)
        results = cursor.fetchall() if fetch else None
        conn.commit()
        print("The Query executed successfully.")
        return results
    except Error as e:
        print(" There was an Error executing query:", e)
    finally:
        close_db(conn)


In [ ]:
# Testing

query = "SELECT first_name, last_name, PIN FROM FOLK LIMIT 5;"
rows = execute_query(query)

if rows:
    for row in rows:
        print(row)


In [ ]:
#Testing insertion:

insert_query = """
INSERT INTO FOLK (PIN, first_name, last_name, nickname, date_of_birth, primary_phone, secondary_phone)
VALUES ('0000000000000004', 'Test', 'Person', 'TP', '1995-07-15', '4105559999', '4105558888');
"""
execute_query(insert_query, fetch=False)

# Verifying insertion
rows = execute_query("SELECT * FROM FOLK WHERE PIN = '0000000000000004';")
for row in rows:
    print(row)

In [ ]:
#Testing update:

update_query = "UPDATE FOLK SET nickname='UpdatedTP' WHERE PIN='0000000000000004';"
execute_query(update_query, fetch=False)

rows = execute_query("SELECT * FROM FOLK WHERE PIN = '0000000000000004';")
for row in rows:
    print(row)

In [ ]:
#Testing deletion:

delete_query = "DELETE FROM FOLK WHERE PIN='0000000000000004';"
execute_query(delete_query, fetch=False)

rows = execute_query("SELECT * FROM FOLK WHERE PIN = '0000000000000004';")
for row in rows:
    print(row)


In [ ]:
def run_sql_file(filepath):
    """Execute all SQL commands from a .sql file safely, including function/procedure blocks."""
    conn = connect_db()
    if not conn:
        return

    try:
        cursor = conn.cursor(dictionary=True)
        with open(filepath, 'r') as f:
            sql_script = f.read()

        lines = sql_script.splitlines()
        statements = []
        current_stmt = []
        in_function_block = False

        for line in lines:
            stripped = line.strip()

            
            if stripped.startswith('--') or stripped == '':
                continue

          
            if stripped.lower().startswith('delimiter'):
                if not in_function_block:
                    in_function_block = True
                    current_stmt = []
                else:
                    # End of the function block
                    statements.append('\n'.join(current_stmt))
                    in_function_block = False
                continue

            if in_function_block:
                current_stmt.append(line)
            else:
                
                if stripped.endswith(';'):
                    statements.append(' '.join(current_stmt + [line]))
                    current_stmt = []
                else:
                    current_stmt.append(line)

        
        for stmt in statements:
            stmt = stmt.strip().rstrip(';')
            if not stmt:
                continue
            try:
                cursor.execute(stmt)
                if cursor.with_rows:
                    rows = cursor.fetchall()
                    for row in rows:
                        print(row)
                else:
                    conn.commit()
                print(f" Executed statement:\n{stmt[:80]}...\n")
            except Error as e:
                print(f" Error executing statement:\n{stmt[:80]}...\n{e}\n")

        print(f" Completed execution of {filepath}")

    except Error as e:
        print(f" Error executing {filepath}:", e)

    finally:
        close_db(conn)




In [ ]:
run_sql_file("queryAll.sql")


In [ ]:
rows = execute_query("SHOW FUNCTION STATUS WHERE Name='GetClosestCenter';")
for row in rows:
    print(row)


In [ ]:
execute_query("""
INSERT INTO OPERATING_PERIOD (center_id, start_dt, end_dt)
VALUES (3, '2025-10-01 00:00:00', '2025-12-01 00:00:00');
""", fetch=False)



In [ ]:
test_query = "SELECT GetClosestCenter('0000000000000001', '2025-11-01') AS closest_center;"
rows = execute_query(test_query)
for row in rows:
    print(row)


In [ ]:
#Let the user input a pin and date:

user_pin = input("Enter a FOLK PIN (e.g. 0000000000000001): ")
user_date = input("Enter a date (YYYY-MM-DD): ")

test_query = f"SELECT GetClosestCenter('{user_pin}', '{user_date}') AS closest_center;"
rows = execute_query(test_query)

for row in rows:
    print(f"Closest Voting Center for {user_pin} on {user_date}: {row['closest_center']}")